In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm

In [ ]:
mni_brain_mask_path = "/path/to/mni_brain_mask.nii.gz"
mni_brain_image_path = "/path/to/mni_template.nii.gz"
mni_ground_truth_dir = "/path/to/mni_registered/ground_truth_masks/folder"
output_dir = "/path/to/output/folder"
cmap = "viridis"                                             # colormap for density
n_slices_per_plane = 20

os.makedirs(output_dir, exist_ok=True)

# load MNI brain mask + background image
mni_mask_img = nib.load(mni_brain_mask_path)
brain_mask = (mni_mask_img.get_fdata() > 0).astype(np.uint8)

mni_bg_img = nib.load(mni_brain_image_path)
mni_bg = mni_bg_img.get_fdata()

# apply mask to background: only brain, set outside to NaN (becomes transparent in PNG)
mni_bg_masked = np.where(brain_mask > 0, mni_bg, np.nan)

# collect all masks
mask_files = sorted(glob.glob(os.path.join(mni_ground_truth_dir, "*_mni_registered_mask.nii.gz")))
if len(mask_files) == 0:
    raise SystemExit(f"No mask files found in {mni_ground_truth_dir} ending with _mni_registered_mask.nii.gz")
print(f"Found {len(mask_files)} mask files.")

# initialize density map
density = np.zeros(brain_mask.shape, dtype=np.int32)

# process masks
for fp in tqdm(mask_files, desc="Processing masks"):
    img = nib.load(fp)
    data = img.get_fdata()

    if img.shape != mni_mask_img.shape:
        print(f"Skipping {fp}: shape mismatch {img.shape} vs {mni_mask_img.shape}")
        continue

    bin_mask = (data > 0).astype(np.uint8)
    clipped = bin_mask * brain_mask
    density += clipped

print(f"Processed {len(mask_files)} masks. Max overlap = {density.max()}")

# ------------------ choose slice indices ------------------
def choose_slice_indices(binary_mask, axis, n_slices):
    nonzero = np.any(binary_mask, axis=tuple(i for i in range(3) if i != axis))
    idxs = np.where(nonzero)[0]
    min_i, max_i = idxs.min(), idxs.max()
    return np.unique(np.round(np.linspace(min_i, max_i, n_slices)).astype(int)).tolist()

sagittal_idxs = choose_slice_indices(brain_mask, axis=0, n_slices=n_slices_per_plane)
coronal_idxs  = choose_slice_indices(brain_mask, axis=1, n_slices=n_slices_per_plane)
axial_idxs    = choose_slice_indices(brain_mask, axis=2, n_slices=n_slices_per_plane)

# ------------------ plotting function ------------------
def save_slice_image(arr_bg, arr_density, plane, idx, cmap="hot"):
    fig, ax = plt.subplots(figsize=(5,5))

    # show anatomical background (NaN -> transparent)
    ax.imshow(np.rot90(arr_bg), cmap="gray", interpolation="nearest")
    
    masked_density = np.where(arr_density>0, arr_density, np.nan)

    # overlay density map with log scale
    if density.max() > 0:
        norm = mcolors.Normalize(vmin=1, vmax=25) 
        im = ax.imshow(np.rot90(masked_density), cmap=cmap, norm=norm, alpha=0.5)
        cb = fig.colorbar(im, ax=ax, shrink=0.6)
        cb.set_label("Mask overlap")

    

    ax.set_title(f"{plane} slice {idx}", fontsize=10)
    ax.axis("off")

    out_file = os.path.join(output_dir, f"{plane}_{idx}.png")
    plt.savefig(out_file, dpi=500, bbox_inches="tight", transparent=True)  # transparency here
    plt.close(fig)

# ------------------ save slices ------------------
# sagittal (x)
for idx in sagittal_idxs:
    save_slice_image(mni_bg_masked[idx, :, :], density[idx, :, :], "sagittal_x", idx, cmap)

# coronal (y)
for idx in coronal_idxs:
    save_slice_image(mni_bg_masked[:, idx, :], density[:, idx, :], "coronal_y", idx, cmap)

# axial (z)
for idx in axial_idxs:
    save_slice_image(mni_bg_masked[:, :, idx], density[:, :, idx], "axial_z", idx, cmap)

print(f"All slice images saved to: {output_dir}")
